In [20]:
import pandas as pd
import geopandas as gpd
import os

In [21]:
# processed 폴더에 저장된 파일 목록 확인
print(os.listdir("../data/processed"))

['dongjak_stations.geojson', 'dongjak_station_accessibility.geojson', 'dongjak_time_risk_layer.geojson']


In [22]:
# 접근성 분석에 필요한 시간대별 위험 레이어와 지하철역 데이터를 불러오기
# 시간대별 Risk Layer
risk_layer_all = gpd.read_file(
    "../data/processed/dongjak_time_risk_layer.geojson"
)

# 데이터 확인
print("Risk Layer 행 수:", len(risk_layer_all))
display(risk_layer_all.head())

Risk Layer 행 수: 16240


,침수이력점수,강수위험점수,최종위험점수,최종위험등급,시간대,geometry
0,1,0,1,낮음,현재,"POLYGON ((949811.827 1944787.218, 949812.697 1..."
1,1,0,1,낮음,현재,"POLYGON ((952917.774 1942919.202, 952923.378 1..."
2,2,0,2,주의,현재,"POLYGON ((954111.346 1943678.468, 954128.274 1..."
3,2,0,2,주의,현재,"POLYGON ((947799.169 1943187.764, 947816.621 1..."
4,2,0,2,주의,현재,"POLYGON ((947743.518 1943193.606, 947745.116 1..."


In [23]:
# Risk Layer의 잘못된 geometry를 다시 보정하고 실제 파일로 저장
from shapely import make_valid

risk_layer_all["geometry"] = risk_layer_all["geometry"].apply(
    lambda geom: make_valid(geom) if not geom.is_valid else geom
)

print(
    "저장 전 Invalid geometry 수:",
    (~risk_layer_all.geometry.is_valid).sum()
)

risk_layer_all.to_file(
    "../data/processed/dongjak_time_risk_layer.geojson",
    driver="GeoJSON",
    encoding="utf-8"
)

print("저장 완료")

저장 전 Invalid geometry 수: 0
저장 완료


In [24]:
# 저장된 GeoJSON을 다시 불러와 geometry 보정이 실제 파일에도 반영됐는지 검증
risk_check = gpd.read_file(
    "../data/processed/dongjak_time_risk_layer.geojson"
)

print("전체 행 수:", len(risk_check))
print(
    "저장 후 Invalid geometry 수:",
    (~risk_check.geometry.is_valid).sum()
)

전체 행 수: 16240
저장 후 Invalid geometry 수: 0


In [25]:
# 저장된 동작구 지하철역 GeoJSON을 새 접근성 분석 노트북으로 불러오기
dongjak_stations = gpd.read_file(
    "../data/processed/dongjak_stations.geojson"
)

print("동작구 지하철역 수:", len(dongjak_stations))
display(dongjak_stations.head())

동작구 지하철역 수: 18


,역사_ID,역사명,호선,위도,경도,침수거리_m,geometry
0,4406,보라매병원,신림선,37.492960,126.923496,234.069920,POINT (949037.795 1943901.321)
1,4405,보라매공원,신림선,37.495569,126.918083,162.926015,POINT (948561.016 1944193.752)
2,4120,동작(현충원),9호선,37.502878,126.978153,583.846343,POINT (953875.509 1944973.495)
3,4119,흑석(중앙대입구),9호선,37.508770,126.963708,79.287174,POINT (952602.474 1945634.355)
4,4118,노들,9호선,37.512887,126.953222,153.708902,POINT (951678.368 1946096.446)


In [26]:
# 서울열린데이터광장 엘리베이터 현황 OpenAPI에서 샘플 5건을 JSON으로 조회
import requests

SEOUL_API_KEY = input("서울열린데이터광장_API_KEY를 입력하세요").strip()

url = (
    f"http://openapi.seoul.go.kr:8088/"
    f"{SEOUL_API_KEY}/json/getWksnElvtr/1/5/"
)

response = requests.get(url, timeout=30)

print("상태코드:", response.status_code)

data = response.json()
print(data)

상태코드: 200
{'response': {'header': {'resultCode': '00', 'resultMsg': 'NORMAL_CODE'}, 'body': {'items': {'item': [{'fcltNo': '1804234', 'fcltNm': '승강기)엘리베이터-종로5가 8,9번 출입구 외부#1', 'lineNm': '1호선', 'stnCd': '0154', 'stnNm': '종로5가', 'stnNo': '129', 'crtrYmd': '20240116', 'mngNo': '외부#1', 'vcntEntrcNo': '내부', 'dtlPstn': '8,9번 출입구 사이', 'bgngFlrGrndUdgdSe': '지하', 'bgngFlr': 'B1', 'endFlrGrndUdgdSe': '지상', 'endFlr': '1', 'pscpNope': 15, 'pscpWht': '1150', 'elvtrSn': '0155-440', 'oprtngSitu': 'M'}, {'fcltNo': '1042124', 'fcltNm': '승강기)엘리베이터-동대문(1) 6번 출구측 외부#1', 'lineNm': '1호선', 'stnCd': '0155', 'stnNm': '동대문', 'stnNo': '128', 'crtrYmd': '20210527', 'mngNo': '외부#1', 'vcntEntrcNo': '6', 'dtlPstn': '6번 출입구', 'bgngFlrGrndUdgdSe': '지하', 'bgngFlr': 'B1', 'endFlrGrndUdgdSe': '지상', 'endFlr': '1', 'pscpNope': 15, 'pscpWht': '1000', 'elvtrSn': '0000-905', 'oprtngSitu': 'M'}, {'fcltNo': '1042125', 'fcltNm': '승강기)엘리베이터-동대문(1) 상행 10-4 내부#1', 'lineNm': '1호선', 'stnCd': '0155', 'stnNm': '동대문', 'stnNo': '128', 'c

In [27]:
# 엘리베이터 API 응답의 실제 시설 목록을 DataFrame으로 변환하고 컬럼을 확인
elevator_df = pd.DataFrame(
    data["response"]["body"]["items"]["item"]
)

print("샘플 행 수:", len(elevator_df))
print("전체 데이터 수:", data["response"]["body"]["totalCount"])

display(elevator_df.head())

샘플 행 수: 5
전체 데이터 수: 865


,fcltNo,fcltNm,lineNm,stnCd,stnNm,stnNo,crtrYmd,mngNo,vcntEntrcNo,dtlPstn,bgngFlrGrndUdgdSe,bgngFlr,endFlrGrndUdgdSe,endFlr,pscpNope,pscpWht,elvtrSn,oprtngSitu
0,1804234,"승강기)엘리베이터-종로5가 8,9번 출입구 외부#1",1호선,0154,종로5가,129,20240116,외부#1,내부,"8,9번 출입구 사이",지하,B1,지상,1,15,1150,0155-440,M
1,1042124,승강기)엘리베이터-동대문(1) 6번 출구측 외부#1,1호선,0155,동대문,128,20210527,외부#1,6,6번 출입구,지하,B1,지상,1,15,1000,0000-905,M
2,1042125,승강기)엘리베이터-동대문(1) 상행 10-4 내부#1,1호선,0155,동대문,128,20210527,내부#1,내부,동묘앞 방면10-4,지하,B2,지하,B1,11,750,0000-906,M
3,1041999,승강기)엘리베이터-동묘앞 본관건물(상)6-2 내부#1,1호선,0159,동묘앞,127,20210527,내부#1,내부,신설동 방면6-2,지하,B1,지상,4,15,1000,0000-946,M
4,1042001,승강기)엘리베이터-동묘앞 상행 4-8 내부#3,1호선,0159,동묘앞,127,20210527,내부#3,내부,신설동 방면4-3,지하,B2,지하,B1,15,1000,0000-947,M


In [28]:
# 동작구 지하철역 이름과 API 역명을 비교하기 위해 괄호 속 부가명칭을 제거한 기준역명 생성
dongjak_stations["기준역명"] = (
    dongjak_stations["역사명"]
    .str.replace(r"\(.*?\)", "", regex=True)
    .str.strip()
)

print(dongjak_stations[["역사명", "기준역명", "호선"]])

           역사명    기준역명   호선
0        보라매병원   보라매병원  신림선
1        보라매공원   보라매공원  신림선
2      동작(현충원)      동작  9호선
3    흑석(중앙대입구)      흑석  9호선
4           노들      노들  9호선
5          노량진     노량진  9호선
6          보라매     보라매  7호선
7       신대방삼거리  신대방삼거리  7호선
8         장승배기    장승배기  7호선
9           상도      상도  7호선
10  숭실대입구(살피재)   숭실대입구  7호선
11          남성      남성  7호선
12          이수      이수  7호선
13         노량진     노량진  경부선
14          사당      사당  4호선
15   총신대입구(이수)   총신대입구  4호선
16     동작(현충원)      동작  4호선
17          사당      사당  2호선


In [29]:
# 엘리베이터 현황 OpenAPI의 전체 865건을 조회해 elevator_df를 다시 생성
url = (
    f"http://openapi.seoul.go.kr:8088/"
    f"{SEOUL_API_KEY}/json/getWksnElvtr/1/865/"
)

response = requests.get(url, timeout=30)
data = response.json()

elevator_df = pd.DataFrame(
    data["response"]["body"]["items"]["item"]
)

print("엘리베이터 전체 행 수:", len(elevator_df))

엘리베이터 전체 행 수: 865


In [30]:
# 엘리베이터 전체 데이터에서 동작구 지하철역과 이름이 일치하는 시설만 추출
dongjak_station_names = dongjak_stations["기준역명"].unique()

dongjak_elevator = elevator_df[
    elevator_df["stnNm"].isin(dongjak_station_names)
].copy()

print("동작구 관련 엘리베이터 수:", len(dongjak_elevator))

display(
    dongjak_elevator[
        [
            "stnNm",
            "lineNm",
            "vcntEntrcNo",
            "dtlPstn",
            "oprtngSitu",
            "elvtrSn"
        ]
    ]
)

동작구 관련 엘리베이터 수: 34


,stnNm,lineNm,vcntEntrcNo,dtlPstn,oprtngSitu,elvtrSn
72,사당,2호선,내부,방배 방면4-4,M,0091-437
93,사당,2호선,14,14번 출입구,M,0091-438
133,사당,2호선,6,6번 출입구,M,0098-381
304,사당,4호선,내부,"총신대입구 방면9-2,남태령 방면2-3",M,0097-112
322,사당,4호선,"9,10","9,10번 출입구 사이",M,0031-517
324,동작,4호선,내부,이촌 방면7-4,M,0097-625
331,동작,4호선,내부,총신대입구 방면4-4,M,0031-516
343,총신대입구,4호선,14,14번 출입구,M,0031-515
350,총신대입구,4호선,1,1번 출입구,M,0110-215
361,총신대입구,4호선,내부,동작 방면7-3,M,0031-512


In [31]:
print("현재 elevator_df 행 수:", len(elevator_df))

현재 elevator_df 행 수: 865


In [32]:
# 동작구 엘리베이터 데이터를 역·호선별로 정리하기 위해 출구연결 여부만 생성
dongjak_elevator["출구연결"] = (
    dongjak_elevator["vcntEntrcNo"] != "내부"
)

display(
    dongjak_elevator[
        [
            "stnNm",
            "lineNm",
            "vcntEntrcNo",
            "dtlPstn",
            "oprtngSitu",
            "출구연결"
        ]
    ].head(20)
)

,stnNm,lineNm,vcntEntrcNo,dtlPstn,oprtngSitu,출구연결
72,사당,2호선,내부,방배 방면4-4,M,False
93,사당,2호선,14,14번 출입구,M,True
133,사당,2호선,6,6번 출입구,M,True
304,사당,4호선,내부,"총신대입구 방면9-2,남태령 방면2-3",M,False
322,사당,4호선,"9,10","9,10번 출입구 사이",M,True
324,동작,4호선,내부,이촌 방면7-4,M,False
331,동작,4호선,내부,총신대입구 방면4-4,M,False
343,총신대입구,4호선,14,14번 출입구,M,True
350,총신대입구,4호선,1,1번 출입구,M,True
361,총신대입구,4호선,내부,동작 방면7-3,M,False


In [33]:
# 역명과 호선별로 엘리베이터 기본 시설정보만 요약해 정적 접근성 데이터 생성
elevator_summary = (
    dongjak_elevator
    .groupby(["stnNm", "lineNm"], as_index=False)
    .agg(
        전체엘리베이터수=("elvtrSn", "nunique"),
        출구연결엘리베이터수=("출구연결", "sum")
    )
)

elevator_summary["출구엘리베이터존재"] = (
    elevator_summary["출구연결엘리베이터수"] > 0
)

display(elevator_summary)

,stnNm,lineNm,전체엘리베이터수,출구연결엘리베이터수,출구엘리베이터존재
0,남성,7호선,4,2,True
1,동작,4호선,2,0,False
2,보라매,7호선,2,1,True
3,사당,2호선,3,2,True
4,사당,4호선,2,1,True
5,상도,7호선,3,1,True
6,숭실대입구,7호선,4,2,True
7,신대방삼거리,7호선,4,1,True
8,이수,7호선,3,1,True
9,장승배기,7호선,3,1,True


In [34]:
# 역별 지하철 데이터와 정적 엘리베이터 시설정보를 결합하고 필요한 컬럼만 확인
station_access = dongjak_stations.copy()

station_access["기준역명"] = (
    station_access["역사명"]
    .str.replace(r"\(.*?\)", "", regex=True)
    .str.strip()
)

station_access = station_access.merge(
    elevator_summary,
    left_on=["기준역명", "호선"],
    right_on=["stnNm", "lineNm"],
    how="left"
)

display(
    station_access[
        [
            "역사명",
            "호선",
            "전체엘리베이터수",
            "출구연결엘리베이터수",
            "출구엘리베이터존재"
        ]
    ]
)

,역사명,호선,전체엘리베이터수,출구연결엘리베이터수,출구엘리베이터존재
0,보라매병원,신림선,NaN,NaN,NaN
1,보라매공원,신림선,NaN,NaN,NaN
2,동작(현충원),9호선,NaN,NaN,NaN
3,흑석(중앙대입구),9호선,NaN,NaN,NaN
4,노들,9호선,NaN,NaN,NaN
5,노량진,9호선,NaN,NaN,NaN
6,보라매,7호선,2.0,1.0,True
7,신대방삼거리,7호선,4.0,1.0,True
8,장승배기,7호선,3.0,1.0,True
9,상도,7호선,3.0,1.0,True


In [35]:
# 엘리베이터 정보의 확인 여부와 출구 접근 가능 여부를 구분해 역별 접근성 상태 생성
# def classify_accessibility(row):
#     if pd.isna(row["전체엘리베이터수"]):
#         return "정보없음"
#     elif row["출구접근가능"] == True:
#         return "접근가능"
#     else:
#         return "확인필요"

# station_access["접근성상태"] = station_access.apply(
#     classify_accessibility,
#     axis=1
# )

# display(
#     station_access[
#         [
#             "역사명",
#             "호선",
#             "전체엘리베이터수",
#             "이용가능엘리베이터수",
#             "출구연결엘리베이터수",
#             "접근성상태"
#         ]
#     ]
# )

In [36]:
# 역별 엘리베이터 접근성 정보를 GeoJSON으로 저장해 개발 단계에서 활용할 수 있도록 준비
station_access.to_file(
    "../data/processed/dongjak_station_accessibility.geojson",
    driver="GeoJSON",
    encoding="utf-8"
)

print("저장 완료")

저장 완료
